# Conditional Analysis 
This notebook shows exemplary functions used to prepare and evaluate our pairwise conditioning approach. 

## Preparation

In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import sys

In [ ]:
suggestive_t=1e-5
# gw_t = 0.05/489673 # for 0.3
gw_t = 0.05/334605 # for 0.2
g_alpha = 0.05/495465
apoe_str_id = 'chr19:44909968:GGGGGGGGGGG:GGGGGGGGG'
apoe_snp_id = '19_45411941_rs429358_T_C'
font_kwargs={
    'fontsize':18
}

In [ ]:
# load str and bellenguez data and extract leading variants
str_data = pd.read_csv('data/strs_biallelic_results.csv', sep="\t")
mask = (str_data.islocmax == 1) & (str_data.P < 1e-5)
str_lead = str_data[mask]

In [ ]:
# load Bellenguez
renamer = {
    'Varianta' : 'ID', 
    'Chromosome' : '#CHROM', 
    'Positionb' : 'POS',
    'P value' :'P_bellenguez'
}
bellenguez_snps = pd.read_csv("path/to/str_gwas/bellenguez_loci.csv", sep="\t").rename(columns=renamer)

In [ ]:
# Find nearest
bellenguez_snps['POS_bellenguez'] = bellenguez_snps['POS']
nearest = pd.merge_asof(
    str_lead.sort_values(by='POS'), 
    bellenguez_snps.sort_values(by='POS'), 
    on='POS', by='#CHROM',
    suffixes=['_STR', '_bellenguez'], 
    direction='nearest', allow_exact_matches=True
)
nearest['dist'] = abs(nearest['POS'] - nearest['POS_bellenguez'])
nearest = nearest.sort_values(by='dist', key=lambda x: abs(x), ascending=False)[
    ['#CHROM', 'POS', 'ID_STR','ID_bellenguez', 'POS_bellenguez', 'dist', 'P', 'P_bellenguez']
    ].rename(columns={'P' : 'P_own'})
nearest.sort_values(by=['#CHROM', 'POS']).head()

In [ ]:
# get same id in our data as well
snp_data = pd.read_csv('data/snp_results.csv', sep="\t")
# bellenguez contains rs numbers, for us its chr_pos_rs
reals = []
for i, r in nearest.iterrows(): 
    id = r['ID_bellenguez']
    new = snp_data[snp_data['ID'].str.contains(f'_{id}_')]['ID']
    id = '-' if new.empty else new.to_string(index=False)
    reals.append(id)

nearest['ID_SNP'] = reals
nearest.head()

In [ ]:
nearest_gw_blz = nearest[(nearest.P_own<gw_t) & (abs(nearest.dist) < 5e5)].sort_values(by=['#CHROM', 'POS'])
pairs = nearest_gw_blz.sort_values(by=['#CHROM', 'POS'])[['ID_STR', 'ID_bellenguez', 'ID_SNP']]

# Add parts around APOE since they are missing in Bellenguez
e4_addon = pd.DataFrame(data=
{
        'ID_STR':['chr19:44909968:GGGGGGGGGGG:GGGGGGGGG'],
        'ID_bellenguez':['-'],
        'ID_SNP':['19_45411941_rs429358_T_C']
}
)

jansen_addon = pd.DataFrame(data=
{
        'ID_STR':['chr19:45802824:CTT:C'],
        'ID_bellenguez':['rs76320948'],
        'ID_SNP':['19_46241841_rs76320948_C_T']
}
)

pairs = pd.concat([pairs, e4_addon, jansen_addon], ignore_index=True)

## Actual runs
They are done on the command line using the run_condition scripts

## Evaluation
**Please note** that the following code is a bit (actually, a lot) messy and was originally not intended for publication. Nevertheless, it does its job. 


### Prep

In [ ]:
adjusted_c = 'darkviolet'
lead_c = 'limegreen'
lead_adjusted_c = 'crimson'
scatter_palette={
    'SNP':'tab:blue', 
    'STR':'tab:orange', 
    'adjusted':'tomato', 
    'unadjusted':'teal'
}
locuszoom_paltette = {
    'SNP':'tab:blue', 
    'STR':'tab:orange', 
    'adjusted': adjusted_c, 
    'unadjusted':'black',
    'STRs adjusted': adjusted_c, 
    'STRs unadjusted':'black',
    'SNPs adjusted': adjusted_c, 
    'SNPs unadjusted':'black',
    'lead STR after conditioning\non Bellengeuz\' SNP' : lead_adjusted_c,
    'lead STR after conditioning on lead SNP' : lead_adjusted_c,
    'lead STR' : lead_c,
    'lead SNP after conditioning\non lead STR' : lead_adjusted_c,
    'lead SNP in Bellenguez/Jansen' : lead_c,
    'lead SNP' : lead_c
}

marker_palette={
    'lead STR after conditioning\non Bellengeuz\' SNP' : "s",
    'lead STR after conditioning on lead SNP' : "s",
    'lead STR' : "D",
    'lead SNP after conditioning\non lead STR' : "s",
    'lead SNP in Bellenguez/Jansen' : "D",
    'lead SNP' : "D"
}

def locus_zoom_review(df, chr, pos, plt_range=5e5, 
               title='', plot_y_thresh=True, highligh_lead=True, suggest_t=5, p_sig_t=-np.log10(1.6*10**(-7)),
               palette=None, style_palette=marker_palette, hue_key='type', kwargs={}, style_key=None, highlight_x=None
    ): 

    if plt_range == 0:
        plt_df = df[(df['#CHROM']==chr)]
    else:
        plt_df = df[(abs(df.POS-pos)<plt_range) & (df['#CHROM']==chr)]
    plt_df = plt_df.sort_values(by='-log10' )
    ax = sns.scatterplot(plt_df, x='POS', y='-log10', hue=hue_key, palette=palette, style=style_key, markers=style_palette, **kwargs)
    
    if plot_y_thresh:
        max_x = plt_df.POS.max()
        min_x = plt_df.POS.min()
        plt.hlines(y=[p_sig_t, suggest_t], xmin=min_x, xmax=max_x, colors=['tab:red', 'k'], linestyles='dashed')
    
    if highligh_lead:
        max_y = plt_df['-log10'].max()
        max_y = ax.get_ylim()[1]*0.97
        if highlight_x is None:
            if plt_df is None or plt_df.empty:
                highlight_x = max_x - plt_range/2
            else: 
                highlight_x = plt_df.iloc[0].POS
                if highlight_x > max_x or highlight_x < min_x:
                    highlight_x = max_x - plt_range/2

        plt.vlines(x=highlight_x, ymin=0, ymax=max_y, alpha=0.1, color='orange', linewidth=30)
    ax.set_ylabel('-log10(p)', **font_kwargs)
    ax.set_xlabel('Genomic coordinate (hg38)', **font_kwargs)

    xticks_df = pd.DataFrame({'POS':[pos - plt_range, pos, pos+plt_range]}).astype('int64')
    ax.set_xticks(xticks_df.POS.to_list())
    ax.set_xticklabels(xticks_df.POS)
    
    plt.title(title)
    plt.grid(axis='y')
    return ax
    
def auto_locuszoom_review(adjusted, initial, chr, pos, marker_id, title='' ,plt_range=3.5e5, newFig=True, legend=True, palette=locuszoom_paltette, key='STRs', ds='own'):
    if newFig:
        plt.figure(figsize=(8,8))
    initial_zoomed = initial[(initial['#CHROM']==chr) & (abs(initial.POS-pos)<plt_range)].copy(deep=True) 
    adjusted_zoomed = adjusted[(adjusted['#CHROM']==chr) & (abs(adjusted.POS-pos)<plt_range)].copy(deep=True) 

    initial_zoomed['type'] = f'{key} unadjusted'
    adjusted_zoomed['type'] = f'{key} adjusted'
    if key == 'STRs':
        initial_zoomed.loc[initial_zoomed.ID==marker_id, 'type'] = 'lead STR'
        if ds == 'own':
            adjusted_zoomed.loc[adjusted_zoomed.ID==marker_id, 'type'] = 'lead STR after conditioning on lead SNP'
        else:
            adjusted_zoomed.loc[adjusted_zoomed.ID==marker_id, 'type'] = 'lead STR after conditioning\non Bellengeuz\' SNP'
    if key == 'SNPs':
        if ds == 'own':
            initial_zoomed.loc[initial_zoomed.ID==marker_id, 'type'] = 'lead SNP'
        else:
            initial_zoomed.loc[initial_zoomed.ID==marker_id, 'type'] = 'lead SNP in Bellenguez/Jansen'
        adjusted_zoomed.loc[adjusted_zoomed.ID==marker_id, 'type'] = 'lead SNP after conditioning\non lead STR'

    if marker_id == None: 
        max_idx= initial_zoomed.P.idxmin()
        marker_id = initial_zoomed.at[max_idx, 'ID']

    combined_zoomed = pd.concat([adjusted_zoomed, initial_zoomed], ignore_index=True).sort_values(by='P')
    combined_zoomed['variant'] = np.where(combined_zoomed.ID == marker_id, marker_id, 'other')

    if palette == None:
        palette={
            'unadjusted':'teal', 
            'adjusted':'tomato',
        }

    ax = locus_zoom_review(combined_zoomed, chr, pos, plt_range=plt_range, hue_key='type', palette=palette, kwargs={'s':30}, highligh_lead=False)
    locus_zoom_review(combined_zoomed[combined_zoomed.variant==marker_id], chr, pos, plt_range=plt_range, palette=palette, hue_key="type", style_key="type", kwargs={'s':100})
    h, l = ax.get_legend_handles_labels()
    ax.legend(h[0:2] + h[4:6], l[0:4], bbox_to_anchor=(1.02, 1), loc=2, borderaxespad=0., fontsize=13)
    if not legend: 
        ax.legend().remove()

    plt.title(title)
    return combined_zoomed.sort_values(by='P')

In [ ]:
def get_results(file_path, is_meta=False, cut=True, cut_threshold=10):
    df = pd.read_csv(file_path, sep="\t")
    initial = df.shape[0]
    df = df.dropna(axis=0)
    after = df.shape[0]
    print(f'drop {initial-after} rows because of NA values')
    df = df.astype({'P':float})
    df.loc[df.P<sys.float_info.min, 'P']=sys.float_info.min

    min_ct = df[df.P == sys.float_info.min].shape[0]
    if min_ct > 0:
        print(f'set {min_ct} P values to maximum possible min of {sys.float_info.min}')
    df['-log10'] = -np.log10(df.P)

    if not is_meta:
        df = df[df['TEST']=='ADD']

    if cut:
        df=df[df['-log10'] < cut_threshold]
    
    return df

In [ ]:
def strip_compare(df, palette=scatter_palette, show_t=True, show_legend=False, title="", plot_break=(20,300), axis_steps = 4):
    
    f, (plot_ax, outlier_ax) = plt.subplots(1, 2, sharey=True, width_ratios=[8,1])
    g = sns.stripplot(df, y='ID', x='-log10', hue='type', legend=show_legend, palette=palette, ax=plot_ax)
    sns.stripplot(df, y='ID', x='-log10', hue='type', legend=False, palette=palette, ax=outlier_ax)
    if show_t:
        y_min, y_max = plt.ylim()
        if y_min < y_max:
            tmp = y_min
            y_min = y_max
            y_max = tmp
        plot_ax.vlines(x=[-np.log10(gw_t), 5], ymax=y_max, ymin=y_min, colors=['tab:red', 'k'], linestyles='dashed')

    outlier_ax.set_xlim(left=plot_break[1])  # outliers only
    plot_ax.set_xlim(0, plot_break[0])  # most of the data
    outlier_ax.spines["left"].set_visible(False)
    plot_ax.spines["right"].set_visible(False)

    # outlier_ax.xaxis.tick_top()
    outlier_ax.tick_params(left=False, right=False)


    step_size = int(plot_break[0] / axis_steps)
    plot_ax.set_xticks(range(0, plot_break[0] + 1, step_size))


    d = .5  # proportion of vertical to horizontal extent of the slanted line
    ax_kwargs = dict(marker=[(-1, -d), (1, d)], markersize=12,
                linestyle="none", color='k', mec='k', mew=1, clip_on=False)
    outlier_ax.plot([0, 0], [0, 1], transform=outlier_ax.transAxes, **ax_kwargs)
    outlier_ax.set_ylabel('')
    plot_ax.plot([1, 1], [1, 0], transform=plot_ax.transAxes, **ax_kwargs)

    g.set_xlabel('-log10(p)', **font_kwargs)
    outlier_ax.set_xlabel("")
    g.set_title(title)
    l = g.get_legend()
    
    sns.move_legend(
        g, 
        loc="lower left",
        bbox_to_anchor=(-.5, -.10),
        # ncol=2, 
        title=None, frameon=True,
    )
    return df[['#CHROM', 'type', 'POS', 'ID', 'P', 'BETA', '-log10']]

### Overview
The following code shows the logic for STRs. For SNP only paths need to be adjusted.

In [ ]:
condi_base = 'path/to/data/conditioned'

In [ ]:
str_data = pd.read_csv('data/strs_biallelic_results.csv', sep="\t")
lead_strs = str_data[str_data.ID.isin(pairs.ID_STR)]
lead_strs['type'] = 'unadjusted'

In [ ]:
adjusted_leads = pd.DataFrame()
call_missings = {}
for i,r in pairs.iterrows():
    str_id = r['ID_STR']
    snp_id = r['ID_SNP']
    new_str_id = str_id.replace('chr', '')
    chr,pos = new_str_id.split(':')[:2]
    file_path = f'{condi_base}/str_runs/{snp_id}_assoc.ad_risk_by_proxy.glm.linear_reduced'
    print(f'running {str_id} adjusted for {snp_id}')
    region_df = get_results(file_path, cut=False)
    if region_df.empty:
        print(f'nothing to do for {snp_id}')
        call_missings[f'{str_id}'] = f'{str_id}*'
        continue
    lstr = region_df[region_df.ID == str_id]
    if lstr.empty:
        call_missings[f'{str_id}'] = f'{str_id}*'
    adjusted_leads = pd.concat([adjusted_leads, lstr])
    continue
adjusted_leads['type'] = 'adjusted'
call_missings

In [ ]:
# Shorten ids for display
replacer = {
    'chr19:44909968:GGGGGGGGGGG:GGGGGGGGG':r'chr19:44909968:G$_{11}$:G$_{9}$',
    'chr15:58846596:AAAAAAAAAAAAAAAAAAAA:AAAAAAAAAAAAAAAAAAA':r'chr15:58846596:A$_{20}$:A$_{19}$', 
    'chr15:63139192:TCCAAAAAAAAAAAAAAAAAAAAAA:TACAAAAAAAAAAAAAAAAAAA': r'chr15:63139192:TCC(A)$_{22}$:TAC(A)$_{19}$'
}
combined_leads = pd.concat([lead_strs, adjusted_leads], ignore_index=True)
combined_leads = combined_leads.replace(replacer)
combined_leads

In [ ]:
_ = strip_compare(combined_leads.sort_values(by=['#CHROM', 'POS']).replace(replacer), plot_break=[20, 300], axis_steps=10, 
                  show_legend=True, title="Comparison of Genomewide significant STRs, \nbefore and after adjustment")
                  

### Per Locus

In [ ]:
# structure is: 
# for each variant in both, STR and bellenguez ID, two analyses were run: 
# 1. 'snp_run' -> on SNP dataset
# 2. 'str_run' -> on STR dataset
# The filename hints the variant ID that was adjusted for
range = 2.5e5
snp_locs, str_locs = [], []
for i,r in pairs.iterrows():
    str_id = r['ID_STR']
    snp_id = r['ID_SNP']
    blz_id = r['ID_bellenguez']

    #run str conditioned for snp
    file_path = f'{condi_base}/str_runs/{snp_id}_assoc.ad_risk_by_proxy.glm.linear_reduced'
    snp_split = snp_id.split('_')
    if len(snp_split) > 1:
        snp_rs = snp_split[2]
    else:
        snp_rs = snp_id
        
    region_df = get_results(file_path, cut=False)
    if region_df.empty:
        print(f'nothing to do for {snp_id}')
        continue
    new_str_id = str_id.replace('chr', '')
    chr,pos = new_str_id.split(':')[:2]
    print(snp_id, chr, pos)
    new_str_id = str_id
    str_pos = int(pos)
    str_locs.append(':'.join([f'chr{chr}', str(str_pos)]))
    if len(new_str_id) > 55:
        new_str_id = new_str_id[:52] + '...'
    title = f'{new_str_id} \nadjusted for \n{snp_rs}'
    if region_df[region_df.ID == str_id].empty:
        title = f'{new_str_id}* \nadjusted for \n{snp_rs}'
    plt.figure(figsize=(16,8))
    plt.subplot(2,1,1)
    
    str_zoom = auto_locuszoom_review(region_df, str_data, int(chr), str_pos, marker_id=str_id, title=title, plt_range=range, newFig=False, ds='bellenguez', key='STRs')
    
    # run snp conditioned for str
    region_df = get_results(f'{condi_base}/snp_runs/{str_id}_assoc.ad_risk_by_proxy.glm.linear_reduced', cut=False)
    if region_df.empty:
        print(f'nothing to do for {str_id}')
        continue
    new_str_id = str_id.replace('chr', '')
    chr,pos = new_str_id.split(':')[:2]
    new_str_id = str_id
    # Here, we need to change it a little since the position in id corresponds to hg37
    title = f'{snp_rs} adjusted for \n{new_str_id}'
    if region_df[region_df.ID == snp_id].empty:
        title = f'SNPs* adjusted for \n{new_str_id}'
    else:
        snp_v = snp_data[snp_data['ID'] == snp_id]
        chr = snp_v.iloc[0]['#CHROM']
        pos = snp_v.iloc[0]['POS']
    snp_locs.append(':'.join([f'chr{chr}', str(pos)]))
    plt.tight_layout()
    plt.subplot(2,1,2)
    # plt.subplot(2,2,4)
    snp_zoom = auto_locuszoom_review(region_df, snp_data, int(chr), str_pos, marker_id=snp_id, title=title, plt_range=range, newFig=False, ds='bellenguez', key='SNPs')
    plt.tight_layout()

    snp_zoom = snp_zoom[['#CHROM', 'POS', 'ID', 'REF', 'ALT', 'A1', 'BETA', 'P', 'type']]
    snp_zoom['source'] = 'SNP'
    str_zoom = str_zoom[['#CHROM', 'POS', 'ID', 'REF', 'ALT', 'A1', 'BETA', 'P', 'type', 'source']]
    str_zoom['source'] = 'STR'

    str_joined = str_zoom[str_zoom.type=='unadjusted'].merge(str_zoom[str_zoom.type=='adjusted'], how='inner', 
            on=['#CHROM', 'POS', 'ID', 'REF', 'ALT', 'A1', 'source'], suffixes=('_unadjusted', '_adjusted')).drop(columns=['type_adjusted', 'type_unadjusted'])
    
    snp_joined = snp_zoom[snp_zoom.type=='unadjusted'].merge(snp_zoom[snp_zoom.type=='adjusted'], how='inner', 
            on=['#CHROM', 'POS', 'ID', 'REF', 'ALT', 'A1', 'source'], suffixes=('_unadjusted', '_adjusted')).drop(columns=['type_adjusted', 'type_unadjusted'])

    if snp_zoom.shape[0]/2 != snp_joined.shape[0]:
        print(f'mismatch on {str_id} snps')
    if str_zoom.shape[0]/2 != str_joined.shape[0]:
        print(f'mismatch on {str_id} strs')

    pd.concat([str_joined, snp_joined]).to_csv(f'{tab_dir}/{str_id}_for_{snp_id}.csv', sep="\t", index=False)

    plt.savefig(f'{fig_dir}/{str_id}_for_{snp_id}.png')